In [5]:
%pip install -q sentence-transformers google-generativeai gradio

import os, re, glob, datetime
import numpy as np
import gradio as gr

print("ready")

ready


In [33]:
NOTES = {
"knowledge/fraud/data_collection.md": """---
type: FraudPattern
title: Excessive & Sneaky Data Collection
---
# Excessive & Sneaky Data Collection
## Data Beyond Need
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Harvesting highly sensitive, unrelated personal logs (SMS, precise coordinates, camera, microphone) is unnecessary for core app functions. This exposes private communications and physical telemetry to third-party servers, creating security breach and blackmail vulnerabilities.
- **Details**: Apps that harvest contacts, SMS, call logs, precise GPS, microphone or camera "for analytics" are over-collecting. A flashlight app wanting contacts = fraud. Modern platforms continuously scan Wi-Fi SSIDs, Bluetooth devices, and signal strengths to reconstruct physical location even if you revoke GPS permissions.

## Bundled Consent
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Forces users into an all-or-nothing trap. It removes granular privacy control, meaning you cannot use essential features without simultaneously agreeing to optional data brokerage.
- **Details**: One "Accept" checkbox that bundles necessary + optional data = invalid consent, red flag. This prevents selective opt-out, forcing users to surrender sensitive telemetry just to use basic tools.

## Background Collection
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Operates as active spyware. By collecting information when idle, companies build unbroken behavior patterns, lifestyle habits, and real-time physical trackers without your knowledge.
- **Details**: Collecting data while the app is closed, or "periodically" uploading device info without an active user session = spyware behavior. This includes silent periodic pings transmitting background logs, accelerometer patterns, and battery temperatures.

## Device Fingerprinting
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Completely bypasses standard browser privacy tools and cookie clearing. It creates an indelible hardware profile that tracks your online movements permanently across factory resets.
- **Details**: Tracking users via device ID, IMEI, MAC address, installed font packages, or browser canvas fingerprint to bypass cookie blocking = sneaky tracking designed to permanently identify devices across factory resets.

## Vague Catch-Alls
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Creates a moving agreement that legally authorizes boundless future data exploitation without requiring updated user approval.
- **Details**: Phrases like "other information we deem necessary" or "as permitted by law" = unlimited data grab. These catch-all phrases grant companies sweeping rights to expand data collection parameters post-facto without notice.""",

"knowledge/fraud/data_sharing.md": """---
type: FraudPattern
title: Data Sharing, Selling & Leaks
---
# Data Sharing, Selling & Leaks
## Selling User Data
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Monetizes your private profile on open marketing exchanges. Once sold, you lose control of who owns, stores, or processes your personal details.
- **Details**: Any clause allowing sale of personal data to "partners" or third parties = DANGEROUS unless fully anonymized and opt-in. Many agreements disguise sales by calling them 'joint marketing arrangements' or 'co-branded opportunities'.

## Unnamed Third Parties
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Conceals the destination of your shared records, making it impossible to request data deletion under regulatory rules like GDPR or CCPA.
- **Details**: "We may share with trusted third parties" without naming them = you can never know who has your data. This allows unlimited automated sharing with advertising exchanges, data brokers, and private contractors.

## Affiliate Loophole
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Circumvents third-party restrictions by circulating your private files inside giant conglomerates.
- **Details**: Defining sister companies, parent firms, or future acquisitions as "not third parties" so data flows freely inside an expansive corporate group without boundaries.

## Change of Ownership
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Treats your private information as an auctionable corporate asset during mergers, buyouts, or bankruptcies.
- **Details**: "We may transfer your data if the company is sold/merged" with no consent required = your data is a saleable asset. This bypasses initial privacy promises since a bankrupt or acquired firm's databases are auctioned off.

## Cross-Border Transfer
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Moves your files to foreign jurisdictions that do not enforce privacy standards, stripping you of practical legal protections.
- **Details**: Moving data to countries with weak privacy laws "to the extent permitted" = leak risk. Once data leaves major jurisdictions like GDPR, recovering or deleting it is practically impossible.

## Ad-Network Sprawl
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Compromises local device security by integrating unvetted external scripts that constantly monitor on-screen activity.
- **Details**: SDKs inside the app share data with hundreds of advertisers/brokers you never agreed to. These SDKs operate on device, matching your app usage history directly with corporate ad-networks.""",

"knowledge/fraud/ai_training.md": """---
type: FraudPattern
title: AI Training on Your Content
---
# AI Training on Your Content
## Stealing Your Content
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Exploits your personal, creative, or proprietary work without compensation, using your intelligence to build models that could eventually replace your job.
- **Details**: "We may use your content to train our AI models" = they take your posts, photos, code, chats forever, royalty-free. This creates derivative intellectual property without user compensation.

## Irrevocable License
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Strips your right to delete your digital legacy. Once trained on, your data cannot be mathematically untangled from deep neural networks.
- **Details**: "You grant us a perpetual, irrevocable, worldwide license to use, reproduce, modify your content" = they own it in practice. Even if you close your account, your creative work remains locked in their models.

## No Opt-Out for AI
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Deprives you of the basic right to declare commercial boundaries while attempting to access mandatory everyday services.
- **Details**: No way to exclude your data from AI training while still using the service = forced surrender. Opt-out controls are often either missing, buried under deceptive UI flows, or disabled entirely.

## Public = Fair Game
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Strips the protective context of public expressions, packaging them into commercial data mines without authorization.
- **Details**: Claiming anything you post publicly can be scraped and reused for any purpose, including commercial AI pipelines. This transforms your public profile into free training data for private profit.""",

"knowledge/fraud/liability.md": """---
type: FraudPattern
title: Liability & Indemnity Traps
---
# Liability & Indemnity Traps
## Total Immunity
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Shifts the legal consequences of corporate negligence, data breaches, and active software bugs onto the consumer.
- **Details**: "Under no circumstances shall we be liable for any damages" even for their own negligence or data breach = harmful, often unenforceable but used to scare users away from seeking legal compensation.

## Cap at Fees Paid
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Prevents actual recovery of damages in severe incidents like identity theft, financial fraud, or structural security leaks.
- **Details**: Limiting all damages to the last $5 you paid even if they leak your identity = unfair. This limits corporate liability to trivial sums regardless of actual real-world damage.

## User Indemnifies Company
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: You are legally forced to fund and defend the company in court, even when the dispute was sparked by their own poor decisions.
- **Details**: "You agree to defend and indemnify us against any claims arising from YOUR use" — you pay their lawyers even when THEY mess up. This is a massive legal trap shifting litigation costs entirely onto the customer.

## Security Breach Disclaimer
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Removes administrative incentives for platforms to maintain up-to-date encryption or strict access protocols.
- **Details**: "We are not responsible for unauthorized access to your data" = they leak it, you suffer, they walk away. This actively disincentivizes companies from investing in modern cybersecurity protocols.""",

"knowledge/fraud/arbitration.md": """---
type: FraudPattern
title: Legal Rights Stripping
---
# Legal Rights Stripping
## Forced Arbitration
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Denies your constitutional right to a public trial before a neutral jury, channeling disputes to private tribunals that statically favor businesses.
- **Details**: Requiring private arbitration and banning courts = you lose public accountability; arbitration favors companies. Arbitration is designed to keep complaints confidential and strip you of standard legal discovery procedures.

## Class-Action Waiver
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Effectively immunizes corporations from accountability for systemic, small-value scams that would cost more to sue individually than the refund is worth.
- **Details**: "You waive the right to join class actions" = you must sue alone, which most people can't afford. This prevents users from pooling resources to fight widespread systematic corporate frauds.

## Unilateral Opt-Out Trap
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Relies on artificial administrative hurdles to secretly lock in forced arbitration before you realize your rights are signed away.
- **Details**: Arbitration opt-out that must be mailed within 30 days in writing = designed so nobody does it. This is a deliberate barrier, relying on user inertia to permanently lock in forced arbitration.

## Governing Law Far Away
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Forces you into unaffordable travel and remote legal representation costs, shutting off practical access to dispute resolution.
- **Details**: Terms governed by law of a distant state/country = suing them costs you a fortune. This forces consumers to hire foreign lawyers and travel thousands of miles to resolve small claims.""",

"knowledge/fraud/unilateral_changes.md": """---
type: FraudPattern
title: Unilateral Changes & Termination
---
# Unilateral Changes & Termination
## Change Anything Anytime
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Allows companies to slip invasive telemetry, higher prices, or predatory legal waivers into agreements after you've integrated the tool.
- **Details**: "We may modify these terms at any time without notice; continued use = acceptance" = moving contract. This allows companies to silently add predatory clauses to an agreement you signed years ago.

## Termination Without Cause
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Threatens your digital identity and operations. You can lose years of work, communications, or bought services with zero leverage or recourse.
- **Details**: Company can delete your account and data anytime without reason or refund. This allows arbitrary bans, permanently destroying your digital presence, documents, or purchased assets.

## No Refund on Kill
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Creates a clear path for subscription fraud, allowing operators to collect annual upfront payments and terminate servers early.
- **Details**: If they shut the service or ban you, "all fees are non-refundable" = they keep your money. They can close down their business tomorrow and keep pre-paid annual subscriptions.

## Delete Your Content
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Enables arbitrary moderation, censorship, and data loss without procedural transparency or appeal.
- **Details**: They may remove your content "at our sole discretion" with no appeal = censorship risk. This strips users of any right to dispute arbitrary content removals or data loss.""",

"knowledge/fraud/subscription.md": """---
type: FraudPattern
title: Subscription & Billing Traps
---
# Subscription & Billing Traps
## Auto-Renewal Trap
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Exploits cognitive inertia and forgetfulness to silently charge payment methods without notice or prior confirmation.
- **Details**: Silent auto-renewal at full price after a cheap trial; cancellation buried 5 menus deep = classic dark pattern. Many apps charge full-price renewals without sending a confirmation email.

## Pre-Ticked Extras
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Artificially inflates transactions by counting on user inattention during fast checkout flows.
- **Details**: Checkout with pre-selected add-ons or "premium" auto-enabled = trick billing. This relies on user inattention during checkout to inflate invoice totals.

## Hard to Cancel
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Employs structural friction (often illegal) to trap user funds inside active subscription loops long after intent has ended.
- **Details**: Requiring phone calls, chat agents, or "retention offers" to cancel while signup took one click = illegal in many places, still common.

## Price Change Rights
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Forces automatic monthly cost updates, removing stable billing guarantees from your long-term budgets.
- **Details**: "We may change pricing at any time with notice" but no refund if you cancel mid-term. This locks users into paying increased rates for current billing cycles.

## No Proration
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Charges you full price for services you've terminated, keeping unearned revenue.
- **Details**: Cancel anytime but "no partial refunds" = they keep money for unused months. They capture payments for services you are no longer utilizing.""",

"knowledge/fraud/tracking.md": """---
type: FraudPattern
title: Tracking, Cookies & Surveillance
---
# Tracking, Cookies & Surveillance
## Track Everywhere
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Constructs highly invasive, cross-website browsing profiles that link financial, social, and professional routines together.
- **Details**: Cookies/pixels following you across other websites and apps, "on and off our services" = cross-site surveillance. This creates a highly invasive profile of your personal browsing habits.

## No Real Opt-Out
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Makes privacy opt-outs a psychological test of endurance, tricking users into standard tracking pathways.
- **Details**: "Do Not Track" ignored; consent banner where "Reject" takes 10 clicks and "Accept" takes 1. Deceptive cookie consent forms use colors and layouts to trick users into accepting tracking.

## Precise Location Forever
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Transforms location history into permanent surveillance logs, posing direct physical security and stalker risks.
- **Details**: Keeping location history indefinitely instead of deleting after use = movement profile = sensitive. This allows companies to pinpoint your home, workplace, and personal routines.

## Selling Ad Profiles
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Fuels unregulated global data broker marketplaces where records are sold to advertisers, insurers, and credit bureaus.
- **Details**: Building and selling advertising profiles combining data from many sources = broker economy leak. Third-party brokers trade profiles linking real names with behavioral histories.

## Microphone/Camera Access
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Poses continuous surveillance risks. Passive devices can capture audio, background environment data, or private visuals.
- **Details**: Ambient listening or background camera access "for features" = hot mic risk. Many apps keep ports open in the background to capture voice commands and context clues.""",

"knowledge/fraud/dark_patterns.md": """---
type: FraudPattern
title: Dark Patterns & Consent Tricks
---
# Dark Patterns & Consent Tricks
## Confirm-Shaming
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Annoying and emotionally manipulative, though it generally doesn't directly compromise data privacy or financial assets.
- **Details**: "Yes, I hate savings" style guilt buttons pressuring acceptance. This uses emotional manipulation to push users into making sub-optimal choices.

## Roach Motel
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Intentionally blocks easy cancellation, keeping you billed against your active will.
- **Details**: Easy in, hard out (sign up in 1 click, cancel via phone call). They intentionally construct asymmetrical UI friction to trap active accounts.

## Sneak Into Basket
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Silently appends hidden fees or extra items directly to checkout balances.
- **Details**: Extra insurance/donation added silently at checkout. These items are appended hoping you don't notice before clicking place order.

## Countdown Timers
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Primarily an artificial sales trick to pressure immediate purchases; does not directly compromise personal files.
- **Details**: Fake urgency ("offer expires in 2 minutes!") that resets = manipulation. This creates anxiety-driven purchase behavior through false scarcity.

## Privacy Zuckering
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Confuses settings arrays to trick users into exposing more contact lists, photos, or GPS tracks than they planned.
- **Details**: Tricking you into sharing more than you intended via confusing settings. Complex layouts disguise data permissions under the banner of 'improving experience'.""",

"knowledge/fraud/children.md": """---
type: FraudPattern
title: Children's Data Traps
---
# Children's Data Traps
## Under-13 Data
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Directly targets impressionable minds, violating children's privacy laws (COPPA) and logging developmental tracks.
- **Details**: Collecting data from children under 13 without parental consent = illegal under COPPA. Sneaky services use child-friendly graphics but collect device identifiers anyways.

## Age Gate Theatre
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Allows platform operators to feign compliance while continuing to collect and trade valuable youth databases.
- **Details**: Just asking "are you 18?" with no verification while clearly marketing to kids. This creates plausible deniability for platforms targeted at minors.

## Targeted Ads to Kids
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Manipulates psychological vulnerabilities in children to drive early-onset commercial consumption.
- **Details**: Behavioral advertising directed at children = predatory. This exploits the psychological vulnerability of minors for financial gain.

## School Data Harvesting
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Converts academic tools into corporate profile catchments that build historical profiles on children from an early age.
- **Details**: Ed-tech collecting student data beyond educational need and reusing it for ads/profiling. This turns mandatory school platforms into corporate profiling tools.""",

"knowledge/fraud/security.md": """---
type: FraudPattern
title: Weak Security & Breach Behavior
---
# Weak Security & Breach Behavior
## No Encryption Promise
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Your passwords, messages, and files are likely stored in unencrypted plain text, accessible to any server intruder or administrative insider.
- **Details**: Terms silent on encryption of stored/transmitted data = assume plaintext. This signals zero technical measures are protecting sensitive communications.

## Password Liability Shift
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Protects corporate negligence in court, holding you responsible for unauthorized purchases even if a database hack leaked your credentials.
- **Details**: "You are responsible for any activity on your account" even if their leak caused it. This protects the business from lawsuit costs if their systems are breached.

## No Breach Notification
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Prevents you from taking swift defensive actions (like rotating compromised passwords) during a leak.
- **Details**: No promise to notify users within a set time if data is breached. GDPR mandates 72h notifications, but global agreements attempt to bypass this with vague schedules.

## Retention Forever
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Long-forgotten servers store decades of sensitive records, preserving perfect historical archives for future leaks.
- **Details**: "We retain your data as long as needed" with no deletion timeline = permanent hoarding. Old accounts contain data that lingers in vulnerable server backends indefinitely.

## Deleted Means Nothing
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Means account cancellation does not fully erase your digital record from target servers.
- **Details**: "We may retain copies in backups" after you delete = your data never truly gone. Backups are kept for years, exposing deleted content during security incidents.""",

"knowledge/fraud/content_ownership.md": """---
type: FraudPattern
title: Content & IP Grabs
---
# Content & IP Grabs
## Royalty-Free Forever
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Permits platforms to monetize, distribute, and license your work without paying you royalties.
- **Details**: License to use your content "royalty-free, perpetual, sublicensable" = they can sell your work. This lets them profit from user creations without paying royalties.

## Moral Rights Waiver
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Allows companies to modify, edit, or remove attribution from your creative work and repurpose it for advertising without your permission.
- **Details**: You waive attribution and integrity rights = they can edit your work and claim it. They are allowed to mutilate your contributions or use them in ads without consent.

## Upload = License
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Claims expansive rights over documents you upload into seemingly private cloud archives.
- **Details**: Merely uploading gives them rights to redistribute your content anywhere, in any media, forever. This applies even if you upload files to 'private' cloud folders.

## Feedback Clause
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Fairly standard software operational clause; you waive royalties on direct software feature suggestions.
- **Details**: Any suggestion you send becomes their property free of charge. They can patent and monetize your ideas without acknowledgment or equity.""",

"knowledge/fraud/software_traps.md": """---
type: FraudPattern
title: Software-Specific Traps
---
# Software-Specific Traps
## Forced Updates
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Can strip out purchased capabilities or lock paid features behind a paywall overnight.
- **Details**: "We may update the software remotely at any time, changing or removing features" = you don't own what you bought. Features can be removed and locked behind paywalls post-purchase.

## Telemetry Always On
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Continuously shares diagnostic, hardware, and runtime files, running down your network and battery limits.
- **Details**: Crash/analytics data sent with no off switch; disabling = features break. This forces a persistent outbound connection, sending diagnostic logs continuously.

## Anti-Reverse-Engineering
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Outlaws security research, meaning independent analysts are legally prohibited from verifying if the program is actively spying.
- **Details**: Banning analysis of what the software does on YOUR device = hides what it collects. This makes it a violation of the terms to discover spyware behavior.

## DRM Lock-In
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Erases access to your purchased library if the company closes down its authentication servers.
- **Details**: Content dies if company shuts down; no export allowed = hostage model. This prevents shifting your purchases to alternative platforms.

## Benchmark Ban
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Primarily acts to prevent open public reviews, though it does not compromise client safety directly.
- **Details**: "Publishing benchmark results is forbidden" = hiding bad performance. This legal ban keeps comparative performance reviews off public forums.

## Spying EULA in "Free" Tools
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Redirects your network traffic through unvetted routing channels to intercept cookies and browse metrics.
- **Details**: Free VPN/proxy apps that log and sell browsing data = the product is you. These tools trade utility for direct network sniffing and tracking.""",

"knowledge/fraud/health_finance.md": """---
type: FraudPattern
title: Sensitive Data Abuse
---
# Sensitive Data Abuse
## Health Data Sharing
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Links intimate medical, biophysical, or biological details with active advertising profiles, creating targets for price discrimination.
- **Details**: Sharing health, fitness, period-tracking data with advertisers = highly sensitive, often illegal. This leaks private medical information directly to marketing engines.

## Financial Data Harvesting
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Collects precise transactional data, fueling aggressive targeted marketing with high-interest loans and predatory pricing.
- **Details**: Reading transaction history or credit info "to improve services" = profiling for predatory offers. This targets you with high-interest loan advertisements.

## Biometric Grab
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Biometric vectors cannot be reset. If a face/fingerprint database leaks, your absolute identity credentials are permanently compromised.
- **Details**: Storing face/voice/fingerprint templates "for convenience" with no deletion = identity theft goldmine. If biometric databases are breached, your unique signatures are permanently compromised.

## Employment Data
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Integrates daily activities with automated scoring brokers who can degrade future employment, credit, or rental ratings.
- **Details**: Selling your usage patterns to data brokers who feed hiring/credit scores. This allows companies to construct corporate profiles influencing employment eligibility.""",

"knowledge/fraud/consent_rights.md": """---
type: FraudPattern
title: Missing User Rights
---
# Missing User Rights
## No Access Right
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Blocks you from verifying the scale of surveillance, allowing companies to silently build and maintain profiles.
- **Details**: No way to see what data they hold on you = GDPR/CCPA violation in many regions. This prevents verifying the scale of tracking.

## No Deletion Right
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Leaves your archives on third-party databases, making it impossible to permanently retire your digital identity.
- **Details**: No process to delete your account and data = retention trap. This locks your historic profile on company servers permanently.

## No Portability
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Locks you into an ecosystem, preventing you from shifting files or records to alternative competitors.
- **Details**: No export of your data in a usable format = lock-in. This makes migrating your networks or databases technically impossible.

## Withdraw Consent Impossible
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Converts a single mouse click into a lifetime surrender of your privacy rights.
- **Details**: Consent given once can never be withdrawn = invalid consent. This makes a single click a permanent surrender of privacy rights.

## Automated Decisions
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Allows untraceable, biased algorithmic decisions to restrict your services without explaining why.
- **Details**: Profiling affecting you (prices, eligibility) with no human review option = rights violation. Algorithmic assessments can penalize you without explanation.""",

"knowledge/fraud/pricing.md": """---
type: FraudPattern
title: Pricing & Fee Frauds
---
# Pricing & Fee Frauds
## Drip Pricing
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Conceals mandatory operational service fees until the last screen, forcing purchasing anxiety and false pricing.
- **Details**: Mandatory "service fees" revealed only at the last step = hidden cost fraud. This artificially deflates advertised prices.

## Negative Option Billing
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Automatically charges your bank card unless you actively navigate a complicated opt-out workflow.
- **Details**: Silence = acceptance of new paid features (you must opt OUT or pay). This automatically charges your account if you ignore update notices.

## Dynamic Personal Pricing
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Uses private context (device brand, zip code, lookup history) to dynamically charge you higher rates.
- **Details**: Prices change based on your device, location, or profile = discrimination. High-end phone users or premium zip codes are systematically shown higher rates.

## Free Trial Charges
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Exploits human forgetfulness to immediately bind you to expensive recurring monthly/annual subscriptions.
- **Details**: Card required for trial, charged instantly at end, reminder only in fine print. This monetizes forgetfulness by enrolling users in annual subscriptions.""",

"knowledge/fraud/communication.md": """---
type: FraudPattern
title: Communication & Spam Clauses
---
# Communication & Spam Clauses
## Marketing Opt-Out Buried
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Clutters inbox folders, but is easily managed with filters and is rarely a financial or systemic safety risk.
- **Details**: Default subscribe to marketing; unsubscribe via postal mail = spam trap. This sets up legal spam systems.

## Selling Contact Info
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Leads directly to aggressive phishing campaigns, scam phone calls, and systemic email tracking.
- **Details**: "We may share your email with select partners" = spam avalanche incoming. Your contact information is packaged and sold as a business asset.

## SMS Consent Grab
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Allows automated text-message spam bots to message your private mobile number, bypassing TCPA safety rules.
- **Details**: "By providing your number you consent to automated marketing texts" hidden mid-paragraph = TCPA violation.

## Push Notification Abuse
- **Danger Level**: 🟢 Safe / Low Risk (Green)
- **Why it is dangerous**: Distracting to device focus, but is simple to disable via native OS permission selectors.
- **Details**: Reserving right to send promotional notifications even after you mute emails. This exploits local device channels to push unprompted ads.""",

"knowledge/fraud/account_control.md": """---
type: FraudPattern
title: Account & Access Control Abuse
---
# Account & Access Control Abuse
## Ban Without Appeal
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Allows companies to terminate your profile, content, and purchased assets without any explanation or recourse.
- **Details**: Account terminated "at our sole discretion," funds/content forfeited, no appeal = kangaroo court. This removes your recourse for errors or biased moderation.

## Device Limit Shenanigans
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Strips true ownership from hardware products, forcing licensing limitations and early hardware obsolescence.
- **Details**: "You may not transfer your license" — can't sell a device with the software = ownership denied.

## Login Surveillance
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Authorizes corporate managers to view your session histories, active devices, and interaction logs without cause.
- **Details**: "We may monitor your account activity at any time" without warrant or suspicion.

## Forced Real Name
- **Danger Level**: 🟡 Caution (Yellow)
- **Why it is dangerous**: Exposes users, commentators, or targets to active doxxing and physical tracking hazards.
- **Details**: Mandatory legal-name policy with public display = doxxing risk. This strips users of their online anonymity and safety protections.

## Two-Factor Paywall
- **Danger Level**: 🔴 Critical Danger (Red)
- **Why it is dangerous**: Treats basic account cybersecurity as an upsell, leaving basic tier users unnecessarily exposed to credential theft.
- **Details**: Basic security (2FA) locked behind premium tier = security as upsell. This leaves basic users exposed unless they pay extra fees.""",
}

for path, text in NOTES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
    print("saved", path)

saved knowledge/fraud/data_collection.md
saved knowledge/fraud/data_sharing.md
saved knowledge/fraud/ai_training.md
saved knowledge/fraud/liability.md
saved knowledge/fraud/arbitration.md
saved knowledge/fraud/unilateral_changes.md
saved knowledge/fraud/subscription.md
saved knowledge/fraud/tracking.md
saved knowledge/fraud/dark_patterns.md
saved knowledge/fraud/children.md
saved knowledge/fraud/security.md
saved knowledge/fraud/content_ownership.md
saved knowledge/fraud/software_traps.md
saved knowledge/fraud/health_finance.md
saved knowledge/fraud/consent_rights.md
saved knowledge/fraud/pricing.md
saved knowledge/fraud/communication.md
saved knowledge/fraud/account_control.md


In [3]:
import glob, os, re

os.makedirs("knowledge", exist_ok=True)
files = [f for f in sorted(glob.glob("knowledge/**/*.md", recursive=True)) if os.path.basename(f) not in ("index.md", "log.md")]
problems = 0
for f in files:
    text = open(f).read()
    missing = []
    if "type:" not in text: missing.append("a type: line")
    if not re.search(r"^# .+", text, re.M): missing.append("a # Title")
    if "## " not in text: missing.append("at least one ## section")
    if missing:
        problems += 1
        print("PROBLEM", f, "is missing", ", ".join(missing))
    else:
        print("OK  ", f)

index = "---\ntype: Index\ntitle: Knowledge Index\n---\n# Index\n" + "".join(
    f"- {f.replace('knowledge/', '')}\n" for f in files)
open("knowledge/index.md", "w").write(index)
print("\nAll notes are good" if problems == 0 else f"\n{problems} note(s) need fixing")
print("saved knowledge/index.md")


All notes are good
saved knowledge/index.md


In [8]:
from sentence_transformers import SentenceTransformer

def load_pieces():
    pieces = []
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True)):
        if os.path.basename(path) in ("index.md", "log.md"):
            continue
        text = open(path).read()
        title = re.search(r"^# (.+)$", text, re.M).group(1)
        for part in text.split("## ")[1:]:
            pieces.append({"file": path,
                           "text": f"{title}: {part.strip()}",
                           "status": "draft" if "status: draft" in text else ""})
    return pieces

def rebuild_index():
    """Reload notes + re-embed. Call this after the knowledge base changes."""
    global PIECES, VECTORS
    PIECES = load_pieces()
    VECTORS = model.encode([p["text"] for p in PIECES], normalize_embeddings=True)

model = SentenceTransformer("all-MiniLM-L6-v2")
rebuild_index()

def find(question):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = VECTORS @ q
    i = int(np.argmax(scores))
    return float(scores[i]), PIECES[i]

print(len(PIECES), "pieces")
score, piece = find("Can I miss classes because I am sick?")
print(round(score, 2), piece["file"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

83 pieces
0.3 knowledge/fraud/arbitration.md


In [16]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
gemini = genai.GenerativeModel("gemini-2.5-flash")

def ask_gemini(question):
    prompt = """You are a careful assistant for a student knowledge base.
Answer the question in 1-3 short, accurate sentences.
If you are not confident, start your reply with "UNSURE:".

Question: """ + question
    try:
        return gemini.generate_content(prompt).text.strip()
    except Exception as e:
        return f"UNSURE: I am currently unable to access Gemini due to an error: {str(e)}"

# quick test (uses a tiny bit of quota)
try:
    print(ask_gemini("What is a black hole, in one sentence?"))
except Exception as e:
    print("API Key loaded, but unable to complete request:", e)

A black hole is a region in spacetime where gravity is so strong that nothing, not even light, can escape from it.


In [18]:
LIMIT = 0.30  # minimum closeness score to trust a note


def agent_core(question):
    score, best = find(question)
    if score >= LIMIT:
        answer = best["text"]
        if best["status"] == "draft":
            answer = "WARNING: this note is a draft - please confirm. " + answer
        return {
            "source": "notes",
            "answer": answer,
            "score": score,
            "file": best["file"],
        }

    # not in notes -> ask Gemini (candidate answer, needs approval)
    answer = ask_gemini(question)
    return {
        "source": "gemini",
        "answer": answer,
        "score": score,
        "file": None,
    }


print(agent_core("What is the attendance requirement?")["source"])  # notes
print(agent_core("What is the canteen menu today?")["source"])  # gemini

gemini
gemini


In [48]:
try:
    import gradio as gr
except ImportError:
    import os
    os.system("pip install -q gradio")
    import gradio as gr

import datetime
import glob
import os
import re

CSS = """
:root {
    --bg-light: #f8fafc;
    --card-bg: #ffffff;
}

body {
    background: radial-gradient(circle at 50% 10%, #eef2ff 0%, #f8fafc 100%) !important;
    overflow-y: auto !important;
}

.gradio-container {
    max-width: 820px !important;
    margin: 0.75rem auto !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    perspective: 1000px;
}

.header-container {
    text-align: center;
    padding: 0.75rem 1rem;
    margin-bottom: 0.75rem;
    background: rgba(255, 255, 255, 0.9);
    backdrop-filter: blur(12px);
    border: 2px solid #e2e8f0;
    border-radius: 20px;
    box-shadow: 0 10px 20px -5px rgba(0, 0, 0, 0.06);
    transform-style: preserve-3d;
    transition: transform 0.4s ease;
    animation: floating3D 5s ease-in-out infinite;
}

.header-container:hover {
    transform: rotateX(3deg) rotateY(-1deg) translateZ(5px);
}

.header-container h1 {
    background: linear-gradient(270deg, #ff0055, #7928ca, #0070f3, #00dfd8, #ff0055);
    background-size: 300% 300%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 900;
    font-size: 1.85rem !important;
    margin: 0 !important;
    animation: rainbowShift 6s linear infinite;
}

@keyframes rainbowShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

@keyframes floating3D {
    0%, 100% { transform: translateY(0px) rotateX(0deg); }
    50% { transform: translateY(-4px) rotateX(1deg); }
}

@keyframes fadeUp {
    from { opacity: 0; transform: translateY(12px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes blink {
    0%, 80%, 100% { opacity: .2; transform: translateY(0); }
    40% { opacity: 1; transform: translateY(-3px); }
}

@keyframes rainbowBorder {
    0% { border-color: #ff0055; box-shadow: 0 0 20px rgba(255, 0, 85, 0.3); }
    25% { border-color: #7928ca; box-shadow: 0 0 20px rgba(121, 40, 202, 0.3); }
    50% { border-color: #0070f3; box-shadow: 0 0 20px rgba(0, 112, 243, 0.3); }
    75% { border-color: #00dfd8; box-shadow: 0 0 20px rgba(0, 223, 216, 0.3); }
    100% { border-color: #ff0055; box-shadow: 0 0 20px rgba(255, 0, 85, 0.3); }
}

.chat-frame {
    border-radius: 24px !important;
    border: 3px solid #ff0055 !important;
    background: #f1f5f9 !important;
    animation: rainbowBorder 6s infinite linear !important;
    padding: 0.6rem !important;
    box-shadow: 0 15px 30px -10px rgba(0, 0, 0, 0.12) !important;
}

#chatbot {
    border-radius: 16px 16px 0 0 !important;
    border: none !important;
    background: #ffffff !important;
    margin-bottom: 0 !important;
}

.input-bar {
    background: #ffffff !important;
    border-top: 2px solid #e2e8f0 !important;
    border-radius: 0 0 16px 16px !important;
    padding: 0.5rem 0.75rem !important;
    align-items: center !important;
    gap: 10px !important;
}

.input-bar textarea {
    border-radius: 24px !important;
    border: 2px solid #0f172a !important;
    background: #ffffff !important;
    padding: 0.5rem 1rem !important;
    font-size: 0.95rem !important;
    font-weight: 500 !important;
    color: #0f172a !important;
    outline: none !important;
    transition: all 0.3s ease !important;
}

.input-bar textarea:focus {
    border-color: #7928ca !important;
    box-shadow: 0 0 0 3px rgba(121, 40, 202, 0.2), 0 3px 10px rgba(121, 40, 202, 0.15) !important;
}

.send-btn {
    border-radius: 50% !important;
    width: 44px !important;
    height: 44px !important;
    min-width: 44px !important;
    max-width: 44px !important;
    padding: 0 !important;
    background: linear-gradient(135deg, #7928ca, #ff0055) !important;
    color: white !important;
    font-size: 1.2rem !important;
    font-weight: bold !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    border: none !important;
    cursor: pointer !important;
    box-shadow: 0 6px 15px rgba(255, 0, 85, 0.3) !important;
    transition: all 0.25s ease !important;
}

.send-btn:hover {
    transform: scale(1.1) rotate(8deg);
    box-shadow: 0 8px 20px rgba(255, 0, 85, 0.45) !important;
}

.msg {
    animation: fadeUp 0.35s ease both;
}

.approval-box {
    border: 2px dashed #7928ca !important;
    border-radius: 16px !important;
    background: rgba(243, 232, 255, 0.9) !important;
    backdrop-filter: blur(8px);
    box-shadow: 0 10px 20px -5px rgba(121, 40, 202, 0.15) !important;
    padding: 1rem !important;
    margin-top: 0.75rem !important;
}

.dot {
    display: inline-block;
    width: 7px;
    height: 7px;
    border-radius: 50%;
    background: #7928ca;
    margin: 0 2px;
    animation: blink 1.2s infinite;
}
.dot:nth-child(2) { animation-delay: .15s; }
.dot:nth-child(3) { animation-delay: .3s; }
"""

def note_files():
    return sorted(
        f
        for f in glob.glob("knowledge/**/*.md", recursive=True)
        if os.path.basename(f) not in ("index.md", "log.md")
    )

def answer_text(res):
    original_ans = res.get("answer", "")

    # Determine status & emojis based on keywords or icons
    if "🔴" in original_ans or "critical" in original_ans.lower() or "danger" in original_ans.lower():
        emoji = "🔴"
        level_text = "Critical Danger!"
    elif "🟡" in original_ans or "caution" in original_ans.lower() or "yellow" in original_ans.lower() or "warning" in original_ans.lower():
        emoji = "🟡"
        level_text = "Caution Required"
    else:
        emoji = "🟢"
        level_text = "Safe & Verified"

    reason_match = re.search(r"Why it is dangerous\*\*:\s*([^\n]+)", original_ans, re.I)
    reason_str = ""
    if reason_match:
        reason_str = f"\n> **⚠️ Why:** {reason_match.group(1)}\n"

    clean_ans = original_ans.replace("-", "").strip()
    formatted_msg = f"### {emoji} {level_text}\n{reason_str}\n{clean_ans}"

    if res.get("source") == "notes":
        file_val = res.get("file", "N/A")
        score_val = res.get("score", 0.0)
        return f"📘 **Document:** `{file_val}` (Score: {score_val:.2f})\n\n{formatted_msg}"
    return formatted_msg

def append_to_note(path, question, answer):
    if not os.path.exists(path):
        title = (
            re.sub(r"\.md$", "", os.path.basename(path))
            .replace("_", " ")
            .title()
        )
        folder = os.path.dirname(path) or "knowledge/misc"
        os.makedirs(folder, exist_ok=True)
        path = os.path.join(folder, os.path.basename(path))
        open(path, "w").write(
            f"---\ntype: Agreement\ntitle: {title}\nsource: fallback\n---\n# {title}\n"
        )
    with open(path, "a") as f:
        f.write(f"\n## {question[:80]}\n{answer}\n")
    with open("knowledge/log.md", "a") as f:
        f.write(
            f"- {datetime.datetime.now():%Y-%m-%d %H:%M} | APPROVED | {path} | {question[:60]}\n"
        )
    if "rebuild_index" in globals():
        rebuild_index()

def ask(question, chat, pending):
    if not question.strip():
        yield chat, pending, gr.update(visible=False), ""
        return
    dots = "<span class='dot'></span><span class='dot'></span><span class='dot'></span>"
    chat = chat + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": dots},
    ]
    yield chat, pending, gr.update(visible=False), ""
    res = agent_core(question.strip())
    chat = chat[:-1] + [
        {
            "role": "assistant",
            "content": f"<div class='msg'>{answer_text(res)}</div>",
        }
    ]
    pending = res if res["source"] == "gemini" else None
    yield chat, pending, gr.update(visible=pending is not None), ""

def approve(pending, target_file, new_file, chat):
    if not pending:
        return chat, None, gr.update(visible=False)
    if target_file == "+ Create new agreement file...":
        fname = new_file.strip().replace(" ", "_").lower() or "agreements"
        path = f"knowledge/agreements/{fname}.md"
    else:
        path = target_file
    append_to_note(path, pending["question_hint"], pending["answer"])
    chat = chat + [
        {
            "role": "assistant",
            "content": f"<div class='msg'>✅ <b>Agreement Saved</b> to <code>{path}</code> — Added to verified knowledge. 🧠</div>",
        }
    ]
    return chat, None, gr.update(visible=False)

def reject(pending, chat):
    if pending:
        chat = chat + [
            {
                "role": "assistant",
                "content": "<div class='msg'>🗑️ <b>Rejected</b> — Draft clause was discarded.</div>",
            }
        ]
    return chat, None, gr.update(visible=False)

def wrap_ask(q, c, p):
    for c2, p2, v, q_clear in ask(q, c, p):
        if p2:
            p2["question_hint"] = q
        yield c2, p2, v, q_clear

with gr.Blocks(title="AgenticRAGreement") as demo:
    with gr.Column(elem_classes=["header-container"]):
        gr.Markdown("# 📜 AgenticRAGreement")

    with gr.Column(elem_classes=["chat-frame"]):
        chatbot = gr.Chatbot(
            label="", height=340, elem_id="chatbot"
        )

        with gr.Row(elem_classes=["input-bar"]):
            qbox = gr.Textbox(
                label="",
                placeholder="Type a message or contract query...",
                container=False,
                scale=9,
                lines=1,
            )
            ask_btn = gr.Button(
                "➢", variant="primary", elem_classes=["send-btn"], scale=1
            )

    pending_s = gr.State(None)

    with gr.Group(visible=False, elem_classes=["approval-box"]) as approval:
        gr.Markdown("### ✋ New Agreement Term Proposed — Approve to integrate into Knowledge Base?")
        target = gr.Dropdown(
            choices=note_files() + ["+ Create new agreement file..."],
            value=note_files()[0] if note_files() else "+ Create new agreement file...",
            label="Select agreement document to update:\n",
        )
        newfile = gr.Textbox(
            visible=False, label="New Agreement Filename", placeholder="nda_vendor_2026.md"
        )
        with gr.Row():
            yes = gr.Button("✅ Approve & Commit", variant="primary")
            no = gr.Button("❌ Reject Draft", variant="stop")

    target.change(
        lambda t: gr.update(visible=t == "+ Create new agreement file..."), target, newfile
    )

    ask_btn.click(
        wrap_ask,
        [qbox, chatbot, pending_s],
        [chatbot, pending_s, approval, qbox],
    )
    qbox.submit(
        wrap_ask,
        [qbox, chatbot, pending_s],
        [chatbot, pending_s, approval, qbox],
    )
    yes.click(
        approve,
        [pending_s, target, newfile, chatbot],
        [chatbot, pending_s, approval],
    )
    no.click(reject, [pending_s, chatbot], [chatbot, pending_s, approval])

    gr.Examples(
        examples=[
            "What is the required termination notice period under our agreement?",
            "What are the confidentiality obligations outlined in the NDA?",
            "What are the payment terms and late payment penalty clauses?",
            "Who holds intellectual property rights for custom software deliverables?",
        ],
        inputs=qbox,
    )

demo.launch(share=True, debug=False, css=CSS, theme=gr.themes.Soft())


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://56f10ea4dd2eb1ab7b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import shutil

shutil.make_archive("my_knowledge", "zip", "knowledge")
try:
    from google.colab import files
    files.download("my_knowledge.zip")
except Exception:
    print("Saved as my_knowledge.zip - grab it from the Files panel (folder icon).")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>